In [1]:
import pandas as pd
import re
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
print("Current folder:")
print(os.getcwd())

print("\nFiles in this folder:")
print(os.listdir())

Current folder:
C:\Users\mouni

Files in this folder:
['.android', '.cache', '.codex', '.copilot', '.eclipse', '.git', '.gitconfig', '.idlerc', '.ipynb_checkpoints', '.ipython', '.jenkins', '.jupyter', '.lesshst', '.m2', '.matplotlib', '.p2', '.python_history', '.ssh', '.VirtualBox', '.vscode', '.vscode-shared', 'AppData', 'Application Data', 'cloud-file-storage', 'Contacts', 'Cookies', 'Documents', 'Downloads', 'eclipse-workspace', 'email-filtering-app', 'Favorites', 'HelloWeb2', 'helloworld', 'IdeaSnapshots', 'index.html', 'IntelGraphicsProfiles', 'kc_house_data.csv', 'Links', 'Local Settings', 'Microsoft', 'Music', 'mvn', 'My Documents', 'mywebapp', 'NetHood', 'ntuser.dat', 'ntuser.dat.log1', 'ntuser.dat.log2', 'NTUSER.DAT{2ad838bc-efea-11ee-a54d-000d3a94eaa1}.TM.blf', 'NTUSER.DAT{2ad838bc-efea-11ee-a54d-000d3a94eaa1}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{2ad838bc-efea-11ee-a54d-000d3a94eaa1}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.dat{c531155a-616f

In [3]:
df = pd.read_csv("spam.csv")

print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [4]:
print(df.columns)
print(df.shape)
print(df["label"].value_counts())

Index(['label', 'message'], dtype='str')
(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64


In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_message"] = df["message"].apply(clean_text)

print(df[["message", "clean_message"]].head())

                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                       clean_message  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in a wkly comp to win fa cup final ...  
3        u dun say so early hor u c already then say  
4  nah i dont think he goes to usf he lives aroun...  


In [6]:
df["label"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

print(df["label"].value_counts())

label
0    4825
1     747
Name: count, dtype: int64


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_message"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4457
Testing samples: 1115


In [8]:
tfidf = TfidfVectorizer(stop_words="english")

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)

TF-IDF training shape: (4457, 7269)


In [9]:
model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

print("Model trained successfully!")

Model trained successfully!


In [10]:
y_pred = model.predict(X_test_tfidf)

print(y_pred[:10])

[0 0 0 1 0 0 0 0 0 0]


In [11]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9641255605381166

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       1.00      0.73      0.84       149

    accuracy                           0.96      1115
   macro avg       0.98      0.87      0.91      1115
weighted avg       0.97      0.96      0.96      1115


Confusion Matrix:
[[966   0]
 [ 40 109]]


In [13]:
messages = [
    "Hey, are you coming to college tomorrow?",
    "Congratulations! You have won a free prize. Click now!",
    "Can you call me when you are free?",
    "URGENT! You have won 10 lakh rupees. Claim your prize now!"
]

messages_tfidf = tfidf.transform(messages)

predictions = model.predict(messages_tfidf)

for message, prediction in zip(messages, predictions):
    if prediction == 1:
        print("SPAM:", message)
    else:
        print("HAM:", message)

HAM: Hey, are you coming to college tomorrow?
SPAM: Congratulations! You have won a free prize. Click now!
HAM: Can you call me when you are free?
SPAM: URGENT! You have won 10 lakh rupees. Claim your prize now!
